<a href="https://colab.research.google.com/github/Shauryasawant/Shauryasawant/blob/main/DDR/RCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers tiktoken mistralai huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-semantic-conventions==0.59b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.


In [ ]:
import os, re, json, math, copy, random, hashlib, time
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from collections import Counter
import numpy as np
import tiktoken
from sentence_transformers import SentenceTransformer
from mistralai.client import Mistral


# ── CELL 3: Auth ─────────────────────────────────────────────
from google.colab import userdata

MISTRAL_API_KEY = userdata.get('token_compress')
HF_TOKEN        = userdata.get('token_compressed')
os.environ['HF_TOKEN']        = HF_TOKEN
os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF authenticated ✓")

TOKENIZER   = tiktoken.get_encoding('cl100k_base')
EMBED_MODEL = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
MISTRAL     = Mistral(api_key=MISTRAL_API_KEY)
CACHE_DIR   = Path('/content/drr_cache')
CACHE_DIR.mkdir(exist_ok=True)
print("Models + Mistral client loaded ✓")


# ================================================================
# PART 1: COMPRESSOR  (sentence-level MMR)
# ================================================================

# ── CELL 4: Compressor Config ────────────────────────────────
@dataclass
class CompressorCfg:
    MMR_LAMBDA:          float = 0.65
    MMR_NOVELTY_W:       float = 0.35
    TARGET_REDUCTION:    float = 0.80
    ARTIFACT_BUDGET_PCT: float = 0.30
    EVIDENCE_FLOOR_PCT:  float = 0.15
    MIN_SENT_TOKENS:     int   = 4
    MAX_EMBED_CHUNK:     int   = 200
    SYSTEM_MAX_TOKENS:   int   = 200
    DECISION_MAX_TOKENS: int   = 60     # cap, not drop, decision-bearing turns
    KEEP_LAST_K:         int   = 2
    PROTECT_TOOL_CALLS:  bool  = True   # never let a tool_call vanish with its content
    PROTECT_JUDGMENTS:   bool  = True   # never let a recommend/confirm turn vanish
    EVIDENCE_BOOST:      float = 0.45   # MMR bonus for sentences a decision actually cites
    ROLE_PRIOR: dict = field(default_factory=lambda: {
        'system':1.10, 'user':1.05, 'assistant':1.00, 'tool':1.00
    })

CCFG = CompressorCfg()

CRITICAL  = {
    'error','exception','warning','fail','success','result','metric','score',
    'recommend','decision','conclusion','path','file','experiment','key','final',
    'found','fix','next','required','critical','important','output','summary'
}
STOPWORDS = {
    'the','and','for','with','that','this','from','into','then','than',
    'what','when','where','read','give','show','tell','make','does','will',
    'would','could','should','have','your','task','same','rate','also',
    'about','while','want','need','please','provide','compare','answer',
    'original','estimate','meaning','just','very','like','more'
}

# ── CELL 5: Compressor Utilities ────────────────────────────
def _tok(text):  return len(TOKENIZER.encode(str(text)))
def _norm(v):    n = np.linalg.norm(v); return v/n if n>0 else v
def _cos(a,b):   return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))

def _encode(texts, cfg=CCFG):
    out = []
    for text in texts:
        toks = TOKENIZER.encode(str(text))
        if len(toks) <= cfg.MAX_EMBED_CHUNK:
            emb = EMBED_MODEL.encode([text], normalize_embeddings=False)[0]
        else:
            step   = cfg.MAX_EMBED_CHUNK - 20
            chunks = [TOKENIZER.decode(toks[i:i+cfg.MAX_EMBED_CHUNK])
                      for i in range(0, len(toks), step)]
            emb = EMBED_MODEL.encode(chunks, normalize_embeddings=False).mean(axis=0)
        out.append(_norm(np.array(emb, dtype=np.float32)))
    return np.vstack(out)

def _get_text(msg):
    parts = []
    for k in ('content','name'):
        v = msg.get(k)
        if isinstance(v, str): parts.append(v)
    tc = msg.get('tool_call')
    if isinstance(tc, dict): parts.append(json.dumps(tc, sort_keys=True))
    return ' '.join(parts)

def _uw(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen: seen.add(x); out.append(x)
    return out

# NOTE: original regex had a stray non-ASCII char (U+017F "long s")
# and broken bracket escaping in the Windows-path branch, which made
# it silently unreliable. Cleaned up here.
_RE_PATH  = re.compile(r'(?:/[\w./-]{3,}|[A-Za-z]:[\\/][\w.\\/-]{3,})')
_RE_URL   = re.compile(r'https?://\S+')
_RE_ID_AZ = re.compile(r'\b[A-Z][A-Z0-9]{1,}-\d+\b')
_RE_ID_HX = re.compile(r'\b[a-f0-9]{8,32}\b')
_RE_NUM   = re.compile(r'\b\d+(?:\.\d+)?%?\b')
_RE_ERR   = re.compile(r'(?i)(?:error|exception|warning)[:\s].{0,120}')
_RE_SENT  = re.compile(r'(?<=[.!?])\s+(?=[A-Z])|(?<=\n)\s*(?=\S)')

def _artifacts(text):
    return {
        'paths':   _uw(_RE_PATH.findall(text)),
        'urls':    _uw(_RE_URL.findall(text)),
        'ids':     _uw(_RE_ID_AZ.findall(text) + _RE_ID_HX.findall(text)),
        'numbers': _uw(_RE_NUM.findall(text)),
        'errors':  _uw(_RE_ERR.findall(text)),
    }

def _art_count(text):
    a = _artifacts(text)
    return sum(len(a[k]) for k in ('paths','urls','ids','errors'))

def _art_bonus(s):
    a = _artifacts(s)
    return min(1.0,
        0.40*min(1.0,len(a['paths']) /1) +
        0.30*min(1.0,len(a['ids'])   /1) +
        0.25*min(1.0,len(a['errors'])/1) +
        0.10*min(1.0,len(a['numbers'])/3)
    )

def _split_sents(text, min_t=4):
    out = []
    for s in _RE_SENT.split(text):
        s = s.strip()
        if not s: continue
        if _art_count(s) > 0 or _tok(s) >= min_t:
            out.append(s)
    return out

def _mmr(sentences, sent_embs, task_emb, budget, art_budget, cfg=CCFG, evidence_artifacts=None):
    if not sentences: return []
    evidence_artifacts = evidence_artifacts or set()

    def _evidence_bonus(s):
        if not evidence_artifacts: return 0.0
        return cfg.EVIDENCE_BOOST if any(a in s for a in evidence_artifacts) else 0.0

    rel = np.array([
        cfg.MMR_LAMBDA * max(0.0, _cos(e, task_emb)) + (1-cfg.MMR_LAMBDA) * _art_bonus(s)
        + _evidence_bonus(s)
        for s, e in zip(sentences, sent_embs)
    ])
    art_idx = sorted(
        [i for i,s in enumerate(sentences) if _art_count(s)>0],
        key=lambda i: rel[i], reverse=True
    )
    chosen, used, art_used = [], 0, 0
    for i in art_idx:
        t = _tok(sentences[i])
        if art_used+t <= art_budget and used+t <= budget:
            chosen.append(i); used+=t; art_used+=t
    pending = sorted([i for i in range(len(sentences)) if i not in chosen],
                     key=lambda i: rel[i], reverse=True)
    while pending and used < budget:
        best_i, best_s = None, -1e18
        for i in pending:
            t = _tok(sentences[i])
            if used+t > budget: continue
            mx = max(_cos(sent_embs[i],sent_embs[j]) for j in chosen) if chosen else 0.0
            sc = rel[i] - cfg.MMR_NOVELTY_W * mx
            if sc > best_s: best_s, best_i = sc, i
        if best_i is None: break
        chosen.append(best_i); used+=_tok(sentences[best_i]); pending.remove(best_i)
    chosen.sort()
    return [sentences[i] for i in chosen]

def _is_decision_bearing(m: Dict, cfg: CompressorCfg) -> bool:
    """A message that, if dropped, deletes a decision rather than
    just trims prose. Tool calls ARE the action decision — if the
    message disappears, so does the tool_call dict riding on it.
    Judgment/confirmation turns are short and evidence-poor by
    nature, so plain relevance scoring tends to lose them to
    evidence-dense tool output competing for the same budget."""
    if cfg.PROTECT_TOOL_CALLS and isinstance(m.get('tool_call'), dict):
        return True
    if cfg.PROTECT_JUDGMENTS and m.get('role') == 'assistant':
        text = _get_text(m)
        if _JUDGMENT_SIGNALS.search(text) or _CONFIRM_SIGNALS.search(text):
            return True
    return False


def compress(messages, cfg=CCFG):
    """Compress a message trace using sentence-level MMR.

    Decision-bearing messages (tool calls, recommendations,
    confirmations) are structurally protected: they always survive,
    capped to DECISION_MAX_TOKENS if long, rather than being subject
    to the same sentence-vs-sentence MMR contest as filler/noise
    turns. Without this, a short instrumental line like "I will
    search for evidence..." reliably loses that contest to
    evidence-dense tool output and the whole message — tool_call
    included — silently disappears.
    """
    task = next((m['content'] for m in messages if m.get('role')=='user'), 'complete the task')
    orig_toks   = sum(_tok(_get_text(m)) for m in messages)
    total_budget = max(1, int(orig_toks * (1-cfg.TARGET_REDUCTION)))
    art_budget   = int(total_budget * cfg.ARTIFACT_BUDGET_PCT)

    n = len(messages)
    protected = set()
    for i,m in enumerate(messages):
        if m.get('role')=='system' or i >= n - cfg.KEEP_LAST_K or _is_decision_bearing(m, cfg):
            protected.add(i)

    task_emb = _encode([task])[0]

    # Evidence boost: find which artifacts are *actually cited* by a
    # real decision (not just "any artifact present anywhere"), so
    # MMR preferentially keeps the evidence a decision depends on
    # over incidental numbers/IDs that no decision ever uses.
    # extract_decisions / build_dependency_graph are defined later in
    # the file but resolved at call time, so this works fine.
    try:
        _decisions_preview = extract_decisions(messages)
        _edges_preview = build_dependency_graph(messages, _decisions_preview)
        evidence_artifacts = {e['artifact'] for e in _edges_preview}
    except NameError:
        evidence_artifacts = set()

    pool: List[Tuple[str,int]] = []
    for i,m in enumerate(messages):
        if i in protected: continue
        for s in _split_sents(_get_text(m), cfg.MIN_SENT_TOKENS):
            pool.append((s, i))

    def _capped(mc, m, max_toks):
        toks = TOKENIZER.encode(m.get('content',''))
        if len(toks) > max_toks:
            mc['content'] = TOKENIZER.decode(toks[:max_toks])+'[…]'
        return mc

    def _protected_toks_with_cap(decision_cap):
        total = 0
        for i in protected:
            m = messages[i]
            cap = cfg.SYSTEM_MAX_TOKENS if m.get('role')=='system' else \
                  (decision_cap if _is_decision_bearing(m, cfg) else None)
            total += min(_tok(_get_text(m)), cap) if cap else _tok(_get_text(m))
        return total

    protected_toks = _protected_toks_with_cap(cfg.DECISION_MAX_TOKENS)
    protected_cap_budget = total_budget * (1 - cfg.EVIDENCE_FLOOR_PCT)
    effective_decision_cap = cfg.DECISION_MAX_TOKENS
    if protected_toks > protected_cap_budget and protected_toks > 0:
        shrink = protected_cap_budget / protected_toks
        effective_decision_cap = max(15, int(cfg.DECISION_MAX_TOKENS * shrink))
        protected_toks = _protected_toks_with_cap(effective_decision_cap)
    remaining = max(total_budget * cfg.EVIDENCE_FLOOR_PCT, total_budget - protected_toks)

    selected_sents = []
    if pool and remaining > 0:
        sents     = [s for s,_ in pool]
        sembs     = _encode(sents)
        selected_sents = _mmr(sents, sembs, task_emb, remaining, art_budget, cfg,
                              evidence_artifacts=evidence_artifacts)

    sent_to_msg = {s:mi for s,mi in pool}
    msg_sents: Dict[int,List[str]] = {}
    for s in selected_sents:
        mi = sent_to_msg.get(s,-1)
        if mi >= 0: msg_sents.setdefault(mi,[]).append(s)

    out = []
    for i,m in enumerate(messages):
        if i in protected:
            mc = dict(m)
            if m.get('role')=='system':
                mc = _capped(mc, m, cfg.SYSTEM_MAX_TOKENS)
            elif _is_decision_bearing(m, cfg):
                mc = _capped(mc, m, effective_decision_cap)   # was cfg.DECISION_MAX_TOKENS
            mc['_orig_idx'] = i
            out.append(mc)
        elif i in msg_sents:
            mc = dict(m)
            mc['content'] = ' '.join(msg_sents[i])
            mc['_orig_idx'] = i
            out.append(mc)
    return out

def compress_identity(messages, **kw):
    """Zero-compression baseline. Passes the trace through unchanged —
    establishes the DRR ceiling: the best score ANY compressor could
    get given the current extractor/reproducer/matcher."""
    return [dict(m, _orig_idx=i) for i, m in enumerate(messages)]

print("Compressor ready ✓")


# ================================================================
# PART 2: DECISION EXTRACTOR
#
# A "decision" is any assistant turn containing:
#   (a) a tool call                          -> action decision
#   (b) a recommendation/conclusion sentence -> judgment decision
#   (c) a confirmation of a prior choice     -> confirmation decision
#
# target is None (not the string 'unknown') when nothing reliable
# can be extracted — match_decision treats None as "unscoreable"
# and excludes it from the average instead of auto-failing it.
# ================================================================

# ── CELL 6: Decision Extractor ───────────────────────────────

_JUDGMENT_SIGNALS = re.compile(
    r'\b(recommend|conclude|suggest|decide|choose|select|prefer|'
    r'best|winner|optimal|final|therefore|thus|hence|confirmed?|'
    r'implement|adopt|use)\b',
    re.IGNORECASE
)
_CONFIRM_SIGNALS = re.compile(
    r'\b(confirm|verified?|preserv|ensur|kept?|maintain)\b',
    re.IGNORECASE
)

# Decisive-verb priority: when several judgment-signal words appear
# in the same turn (e.g. "...clear winner... Recommend implementing
# ..."), prefer the verb that actually states the decision over a
# descriptive noun like "winner" or "best".
_ACTION_VERB_PRIORITY = [
    'recommend', 'confirm', 'adopt', 'implement', 'select', 'choose',
    'decide', 'prefer', 'suggest', 'conclude', 'use',
    'therefore', 'thus', 'hence', 'best', 'optimal', 'winner', 'final',
]

_ENTITY_BLOCKLIST = CRITICAL | STOPWORDS | {
    'evidence', 'confirmed', 'confirm', 'confirmation', 'confirms',
    'winner', 'best', 'optimal', 'reading', 'comparing', 'report',
    'results', 'recommendation', 'recommended', 'implementing',
    'preserved', 'clear', 'metrics', 'detailed',
}
_RE_ENTITY = re.compile(r'\b[A-Z][A-Za-z0-9]{2,}\b')


def _cap_target_length(target: Optional[str], max_words: int = 6) -> Optional[str]:
    """A target derived from a free-text arg (e.g. a search `query`)
    can end up being the entire task sentence — no reproducer would
    ever guess that back verbatim, so it's not a fair ground truth.
    Null it out instead of penalising every reproduction for it."""
    if not target:
        return target
    return target if len(target.split()) <= max_words else None


def _extract_entity_target(text: str, known_ids: Optional[List[str]] = None) -> Optional[str]:
    """Most-frequent capitalised entity-like token that isn't a
    discourse word and isn't a prefix of a known ID (e.g. 'EXP' is a
    prefix of 'EXP-2824' and is excluded so it can't win by
    coincidence, letting the actual named entity — e.g. a method
    name — surface instead)."""
    known_ids = known_ids or []
    id_prefixes = {i.split('-')[0] for i in known_ids if '-' in i}
    candidates = [
        w for w in _RE_ENTITY.findall(text)
        if w.lower() not in _ENTITY_BLOCKLIST and w not in id_prefixes
    ]
    if not candidates:
        return None
    return Counter(candidates).most_common(1)[0][0]


def _extract_target(args: dict, text: str) -> Optional[str]:
    """What the decision is ABOUT. Priority: explicit tool arg >
    named entity actually being judged > artifact ID/path as a last
    resort. Returns None (not 'unknown') when nothing reliable is
    found.

    'options' is included alongside the single-value keys: a
    'compare' call's target is the *set* it's comparing, not one
    arbitrary name plucked out of the tool_call JSON dump by the
    entity-guess fallback below. Without this, ground truth for
    'compare' was effectively random (whichever capitalised token
    serialised first), so it failed reproduction even at zero
    compression — an extraction bug, not a compression failure."""
    for k in ('paper_id', 'file', 'path', 'query', 'id', 'target', 'name', 'options'):
        if k in args:
            v = args[k]
            t = json.dumps(v) if isinstance(v, (list, dict)) else str(v)
            return _cap_target_length(t)
    arts = _artifacts(text)
    ent = _extract_entity_target(text, known_ids=arts['ids'])
    if ent:
        return ent
    for k in ('ids', 'paths'):
        if arts[k]:
            return arts[k][0]
    return None


def _extract_verb(text: str) -> str:
    matches = [g.lower() for g in _JUDGMENT_SIGNALS.findall(text)]
    if not matches:
        return 'decide'
    for v in _ACTION_VERB_PRIORITY:
        if v in matches:
            return v
    return matches[0]


def _extract_rationale(text: str, arts: dict) -> List[str]:
    """Evidence cited in this decision: numbers/metrics, errors,
    named IDs."""
    rationale = []
    for m in re.finditer(r'([A-Za-z][\w-]*)\s*[=:]\s*(\d+(?:\.\d+)?%?)', text):
        rationale.append(f'{m.group(1)}={m.group(2)}')
    for e in arts['errors'][:2]:
        rationale.append(f'error:{e[:60]}')
    for i in arts['ids'][:3]:
        rationale.append(f'id:{i}')
    return rationale


def extract_decisions(messages: List[Dict]) -> List[Dict]:
    """Extract structured decisions from all assistant turns."""
    decisions = []
    for idx, msg in enumerate(messages):
        if msg.get('role') != 'assistant':
            continue
        text = _get_text(msg)
        arts = _artifacts(text)

        if isinstance(msg.get('tool_call'), dict):
            tc = msg['tool_call']
            decisions.append({
                'type':      'action',
                'action':    tc.get('name', 'unknown_tool'),
                'target':    _extract_target(tc.get('args', {}), text),
                'rationale': _extract_rationale(text, arts),
                'artifacts': {k: arts[k] for k in ('paths', 'ids', 'errors')},
                'verbatim':  text,
                'msg_idx':   idx,
            })
        elif _JUDGMENT_SIGNALS.search(text):
            decisions.append({
                'type':      'judgment',
                'action':    _extract_verb(text),
                'target':    _extract_target({}, text),
                'rationale': _extract_rationale(text, arts),
                'artifacts': {k: arts[k] for k in ('paths', 'ids', 'errors')},
                'verbatim':  text,
                'msg_idx':   idx,
            })
        elif _CONFIRM_SIGNALS.search(text) and (arts['paths'] or arts['ids']):
            target = _extract_target({}, text)
            if target is None:
                fallback = (arts['paths'] + arts['ids'])
                target = fallback[0] if fallback else None
            decisions.append({
                'type':      'confirmation',
                'action':    'confirm',
                'target':    target,
                'rationale': _extract_rationale(text, arts),
                'artifacts': {k: arts[k] for k in ('paths', 'ids', 'errors')},
                'verbatim':  text,
                'msg_idx':   idx,
            })

    return decisions

print("Decision extractor ready ✓")


# ================================================================
# PART 3: DECISION REPRODUCER
#
# Given a compressed trace, ask Mistral to reproduce each decision
# cold. The reproducer is told the trace's actual tool-call
# vocabulary, so it can recover an internal name like "compare"
# instead of guessing an English synonym for it.
# ================================================================

# ── CELL 7: Decision Reproducer ──────────────────────────────

REPRODUCE_SYSTEM = """\
You are a decision auditor for AI agent traces.
Given a compressed agent trace and a step index, your job is to
identify what decision the agent made at that step.

IMPORTANT:
- Base your answer ONLY on the compressed trace provided.
- Return a single JSON object — no other text.
- If you cannot determine a field from the trace, use null.

JSON schema:
{
  "action":    string,   // verb: what did the agent do? (e.g. "web_search", "recommend", "confirm")
  "target":    string,   // what was the object? (e.g. "CompressMMR", "EXP-2401", "Paper A")
  "rationale": [string], // list of evidence keys cited (e.g. ["SP=0.91", "task-success=82%"])
  "confidence": float    // 0.0-1.0: how certain are you given the compressed trace?
}
"""

def _format_trace_for_prompt(messages: List[Dict]) -> str:
    lines = []
    for pos, m in enumerate(messages):
        idx = m.get('_orig_idx', pos)   # <-- use true original index, not list position
        role = m.get('role', '?').upper()
        text = _get_text(m)
        tc   = m.get('tool_call')
        if tc:
            lines.append(f'[{idx}] {role}: {text[:400]} → TOOL:{tc.get("name","")}({json.dumps(tc.get("args",{}))[:200]})')
        else:
            lines.append(f'[{idx}] {role}: {text[:400]}')
    return '\n'.join(lines)

def _build_tool_catalogue(messages: List[Dict]) -> List[str]:
    names = []
    for m in messages:
        tc = m.get('tool_call')
        if isinstance(tc, dict) and tc.get('name'):
            names.append(tc['name'])
    return _uw(names)

def reproduce_decision(
    compressed_msgs: List[Dict],
    decision:        Dict,
    model:           str = 'mistral-small-latest',
    max_retries:     int = 2,
) -> Dict:
    """Ask Mistral to reproduce a specific decision from the
    compressed trace, cold."""
    tools = _build_tool_catalogue(compressed_msgs)
    tool_line = (
        f"\nKnown tool names this agent can call: {', '.join(tools)}. "
        f"If the decision was a tool call, 'action' MUST be one of these exact strings."
        if tools else ""
    )
    trace_text = _format_trace_for_prompt(compressed_msgs)
    trace_text = (
        "NOTE: step numbers below may have gaps — only the listed [N] rows exist.\n"
        + trace_text
    )

    type_hint = {
        'action':       "This was a TOOL CALL. 'action' must be the exact tool name "
                        "(see the known tool names below). 'target' is what the tool "
                        "was applied to, if determinable.",
        'judgment':     "This was a JUDGMENT/RECOMMENDATION. 'action' should be the "
                        "single decisive verb (e.g. 'recommend', 'select', 'confirm'), "
                        "not a descriptive noun like 'winner' or 'best'. 'target' is "
                        "the specific option/entity being judged.",
        'confirmation': "This turn CONFIRMED a prior choice. 'action' should be "
                        "'confirm'. 'target' is what was confirmed.",
    }.get(decision['type'], "")

    step_hint  = (
        f"Step {decision['msg_idx']} was an assistant '{decision['type']}' turn. "
        f"{type_hint} What did the agent decide to do at this step?"
    )

    prompt = f"""COMPRESSED AGENT TRACE:
{trace_text}
{tool_line}

TASK:
{step_hint}

Return ONLY a JSON object with keys: action, target, rationale (list), confidence."""

    for attempt in range(max_retries + 1):
        try:
            resp = MISTRAL.chat.complete(
                model=model,
                messages=[
                    {'role':'system',  'content': REPRODUCE_SYSTEM},
                    {'role':'user',    'content': prompt},
                ],
                temperature=0.0,
                max_tokens=300,
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw)
            result = json.loads(raw)
            result['_raw']     = raw
            result['_success'] = True
            return result
        except (json.JSONDecodeError, Exception) as e:
            if attempt == max_retries:
                return {
                    'action': None, 'target': None,
                    'rationale': [], 'confidence': 0.0,
                    '_raw': str(e), '_success': False,
                }
            time.sleep(1)

print("Decision reproducer ready ✓")


# ================================================================
# PART 4: DECISION MATCHER
#
# Structured field matching with partial credit — not string
# equality (too strict), not embedding similarity (too loose, the
# same problem DRR exists to fix).
#
#   decision_score = 0.40*action_match + 0.40*target_match + 0.20*rationale_F1
#
# A None ground-truth target is "unscoreable by design": it's
# excluded from the average and its weight is redistributed onto
# action + rationale, instead of silently forcing a 0.
# ================================================================

# ── CELL 8: Decision Matcher ─────────────────────────────────

W_ACTION      = 0.40
W_TARGET      = 0.40
W_RATIONALE   = 0.20
DRR_THRESHOLD = 0.75

def _normalise(text: str) -> str:
    if text is None: return ''
    return re.sub(r'[^a-z0-9]', ' ', str(text).lower()).strip()

def _token_f1(pred: str, gold: str) -> float:
    p_toks = set(_normalise(pred).split())
    g_toks = set(_normalise(gold).split())
    if not p_toks or not g_toks: return 0.0
    overlap = len(p_toks & g_toks)
    prec    = overlap / len(p_toks)
    rec     = overlap / len(g_toks)
    return 2*prec*rec/(prec+rec+1e-9)

def _stem(word: str) -> str:
    """Light suffix-stripping so 'searched'/'searching'/'search' and
    'recommended'/'recommends'/'recommend' compare equal without
    pulling in a real stemmer dependency."""
    for suf in ('ing', 'ed', 's'):
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[:-len(suf)]
    return word

def _action_match(original: Dict, reproduced: Dict) -> float:
    orig = _normalise(original.get('action',''))
    repr_ = _normalise(reproduced.get('action',''))
    if not orig or not repr_: return 0.0
    if orig == repr_: return 1.0
    # Word-boundary stem match: each word of the shorter side must
    # appear (after stemming) as a whole word in the longer side.
    # Plain substring ("search" in "researching") gave false
    # positives/negatives; this is closer to the true comparison.
    orig_stems  = {_stem(w) for w in orig.split()}
    repr_stems  = {_stem(w) for w in repr_.split()}
    if orig_stems and orig_stems <= repr_stems: return 0.9
    if repr_stems and repr_stems <= orig_stems: return 0.8
    if orig_stems & repr_stems: return 0.6
    return _token_f1(repr_, orig)

def _target_match(original: Dict, reproduced: Dict) -> float:
    orig  = original.get('target','')
    repr_ = reproduced.get('target','')
    if isinstance(orig, list):  orig  = ' '.join(str(x) for x in orig)
    if isinstance(repr_, list): repr_ = ' '.join(str(x) for x in repr_)
    if not orig or not repr_: return 0.0
    orig  = _normalise(orig)
    repr_ = _normalise(repr_)
    if orig == repr_: return 1.0
    return _token_f1(repr_, orig)

def _rationale_f1(original: Dict, reproduced: Dict) -> float:
    orig_rat = [_normalise(r) for r in original.get('rationale',[]) if r]
    repr_rat = [_normalise(r) for r in reproduced.get('rationale',[]) if r]
    if not orig_rat: return 1.0
    if not repr_rat: return 0.0
    hits = 0.0
    for og in orig_rat:
        best = max((_token_f1(rp, og) for rp in repr_rat), default=0.0)
        hits += best
    return hits / len(orig_rat)

def match_decision(original: Dict, reproduced: Dict) -> Dict:
    """Compare one original decision with its reproduced counterpart."""
    if not reproduced.get('_success', True):
        return {
            'action_score': 0.0, 'target_score': 0.0, 'rationale_score': 0.0,
            'decision_score': 0.0, 'reproduced': False, 'failure': 'llm_error',
            'target_scoreable': original.get('target') is not None,
        }

    a_score = _action_match(original, reproduced)
    r_score = _rationale_f1(original, reproduced)
    target_scoreable = original.get('target') is not None

    if target_scoreable:
        t_score = _target_match(original, reproduced)
        overall = W_ACTION*a_score + W_TARGET*t_score + W_RATIONALE*r_score
    else:
        t_score = None
        overall = (W_ACTION/(W_ACTION+W_RATIONALE))*a_score + (W_RATIONALE/(W_ACTION+W_RATIONALE))*r_score

    return {
        'action_score':     round(a_score, 4),
        'target_score':     None if t_score is None else round(t_score, 4),
        'rationale_score':  round(r_score, 4),
        'decision_score':   round(overall, 4),
        'reproduced':       overall >= DRR_THRESHOLD,
        'failure':          None,
        'target_scoreable': target_scoreable,
    }

print("Decision matcher ready ✓")


# ================================================================
# PART 5: DRR PIPELINE
# ================================================================

# ── CELL 9: Legacy Metrics (kept for comparison against DRR) ──

def _legacy_metrics(orig_msgs, comp_msgs):
    orig_text = ' '.join(_get_text(m) for m in orig_msgs)
    comp_text = ' '.join(_get_text(m) for m in comp_msgs)
    oe, ce = _encode([orig_text, comp_text])
    SP     = _cos(oe, ce)

    def ug(t): return Counter(re.findall(r'\b[a-z]{2,}\b', t.lower()))
    o,c = ug(orig_text), ug(comp_text)
    hits = sum(min(o[w],c[w]) for w in o if w in c)
    R1   = hits / max(1, sum(o.values()))

    orig_arts = _artifacts(orig_text)
    total = retained = 0
    for k in ('paths','ids','errors'):
        for item in orig_arts[k]:
            total += 1
            if item in comp_text: retained += 1

    return {
        'SP':      round(SP, 4),
        'ROUGE1':  round(R1, 4),
        'art_ret': round(retained/max(1,total), 4),
    }

# ── CELL 10: Single-Trace DRR ───────────────────────────────

def compute_drr(
    messages:     List[Dict],
    compressor=compress,
    compress_cfg: CompressorCfg = CCFG,
    verbose:      bool          = True,
) -> Dict:
    """Full DRR pipeline for a single trace. `compressor` defaults
    to your real compressor; pass `compress_identity` to get the
    DRR ceiling (zero compression) for comparison."""
    if verbose:
        print('='*65)
        print('DECISION REPRODUCIBILITY COMPRESSION')
        print('='*65)

    decisions = extract_decisions(messages)
    if verbose:
        print(f'\n[1] Decisions extracted: {len(decisions)}')
        for d in decisions:
            print(f'    [{d["type"]:12s}] {d["action"]:12s} → target={str(d["target"])[:40]}')

    if not decisions:
        if verbose: print('    No decisions found — DRR undefined')
        return {'DRR_soft':None,'DRR_binary':None,'decisions':[],'compressed':messages}

    if verbose: print('\n[2] Compressing trace...')
    compressed = compressor(messages, cfg=compress_cfg) if compressor is compress else compressor(messages)
    orig_toks  = sum(_tok(_get_text(m)) for m in messages)
    comp_toks  = sum(_tok(_get_text(m)) for m in compressed)
    reduction  = 100*(1 - comp_toks/max(1,orig_toks))
    if verbose:
        print(f'    {orig_toks} → {comp_toks} tokens  ({reduction:.1f}% reduction)')

    if verbose: print('\n[3] Reproducing decisions from compressed trace...')
    results = []
    for i, decision in enumerate(decisions):
        reproduced = reproduce_decision(compressed, decision)
        match      = match_decision(decision, reproduced)
        results.append({'decision_idx': i, 'original': decision, 'reproduced': reproduced, 'match': match})
        if verbose:
            flag = '✓' if match['reproduced'] else '✗'
            ts = 'n/a' if match['target_score'] is None else f'{match["target_score"]:.2f}'
            print(f'    [{flag}] Decision {i} ({decision["type"]:12s}): '
                  f'score={match["decision_score"]:.3f}  '
                  f'action={match["action_score"]:.2f}  target={ts}  '
                  f'rationale={match["rationale_score"]:.2f}')

    scores   = [r['match']['decision_score'] for r in results]
    DRR_soft = float(np.mean(scores)) if scores else 0.0
    DRR_bin  = float(np.mean([r['match']['reproduced'] for r in results])) if results else 0.0
    legacy   = _legacy_metrics(messages, compressed)
    n_unscoreable = sum(1 for r in results if not r['match']['target_scoreable'])

    if verbose:
        print(f"""
[4] Results
  DRR  (soft, mean score)    : {DRR_soft:.4f}
  DRR  (binary, ≥{DRR_THRESHOLD:.2f})      : {DRR_bin:.2%}
  Decisions total / reproduced: {len(decisions)} / {sum(r["match"]["reproduced"] for r in results)}
  Decisions with unscoreable target (excluded from target_score): {n_unscoreable}/{len(decisions)}
  Reduction                  : {reduction:.1f}%
  Semantic pres. (SP)        : {legacy["SP"]:.4f}
  ROUGE-1 recall             : {legacy["ROUGE1"]:.4f}
  Artifact retention         : {legacy["art_ret"]:.2%}
""")

    return {
        'DRR_soft': DRR_soft, 'DRR_binary': DRR_bin, 'decisions': results,
        'compressed': compressed, 'orig_tokens': orig_toks, 'comp_tokens': comp_toks,
        'reduction': reduction, 'n_unscoreable_target': n_unscoreable, **legacy,
    }

print("DRR pipeline ready ✓")


# ================================================================
# PART 6: SYNTHETIC TRACE GENERATOR
# ================================================================

# ── CELL 11: Synthetic Trace Generator ───────────────────────

TASKS = [
    {'task':'Choose the best compression algorithm for agent traces.',
     'options':['CompressMMR','BudgetComp','SelectiveCtx'],
     'winner':'CompressMMR',
     'winner_metrics':{'SP':0.91,'task_success':0.82,'compression':0.73},
     'loser_metrics':[
         {'SP':0.88,'task_success':0.77,'compression':0.68},
         {'SP':0.84,'task_success':0.74,'compression':0.50},
     ]},
    {'task':'Select the best embedding model for sentence scoring.',
     'options':['all-MiniLM-L6-v2','all-mpnet-base-v2','paraphrase-MiniLM'],
     'winner':'all-mpnet-base-v2',
     'winner_metrics':{'F1':0.89,'latency_ms':84,'dim':768},
     'loser_metrics':[
         {'F1':0.83,'latency_ms':45,'dim':384},
         {'F1':0.79,'latency_ms':32,'dim':384},
     ]},
    {'task':'Identify the root cause of the pipeline failure.',
     'options':['OOM error','network timeout','corrupt checkpoint'],
     'winner':'corrupt checkpoint',
     'winner_metrics':{'error_log':'checkpoint_load_failed','file':'/checkpoints/run_42.pt'},
     'loser_metrics':[
         {'error_log':'none'},
         {'error_log':'transient'},
     ]},
    {'task':'Choose the data preprocessing strategy.',
     'options':['tokenise_first','chunk_then_embed','embed_raw'],
     'winner':'chunk_then_embed',
     'winner_metrics':{'ROUGE':0.76,'memory_mb':512,'throughput':340},
     'loser_metrics':[
         {'ROUGE':0.71,'memory_mb':128,'throughput':890},
         {'ROUGE':0.68,'memory_mb':96,'throughput':1200},
     ]},
    {'task':'Select the retrieval strategy for RAG pipeline.',
     'options':['BM25','dense_retrieval','hybrid'],
     'winner':'hybrid',
     'winner_metrics':{'MRR':0.84,'latency_ms':120,'P@10':0.79},
     'loser_metrics':[
         {'MRR':0.71,'latency_ms':45,'P@10':0.65},
         {'MRR':0.78,'latency_ms':95,'P@10':0.73},
     ]},
]

NOISE_TEMPLATES = [
    'API metadata: region=us-east-1 latency={lat}ms call_id={cid} rate_limit={rl}/1000.',
    'Cache miss for key {cid}. Fetching from origin. TTL=3600s.',
    'Pagination: page={pg} of {total}. Cursor={cid}. Next batch in {lat}ms.',
    'Telemetry: span_id={cid} trace_id={cid2} service=agent version=2.1.4.',
    'Rate limit status: {rl}/1000 remaining. Reset in {lat}s. Bucket=default.',
    'Index version: {cid}. Shard 3 of 8. Replica lag: {lat}ms.',
]

def _make_noise(n=3):
    lines = []
    rng = random.Random(hashlib.md5(str(n).encode()).hexdigest())
    for tmpl in rng.sample(NOISE_TEMPLATES, min(n, len(NOISE_TEMPLATES))):
        lines.append(tmpl.format(
            lat=rng.randint(10,500), cid=hashlib.md5(str(rng.random()).encode()).hexdigest()[:8],
            rl=rng.randint(500,999), pg=rng.randint(1,10), total=rng.randint(10,50),
            cid2=hashlib.md5(str(rng.random()).encode()).hexdigest()[:8],
        ))
    return ' '.join(lines)

def generate_trace(task_spec: dict, noise_level: int = 3, rng_seed: int = 0) -> List[Dict]:
    """Generate a synthetic multi-turn agent trace for a task spec."""
    rng   = random.Random(rng_seed)
    opts  = task_spec['options']
    win   = task_spec['winner']
    win_m = task_spec['winner_metrics']
    los_m = task_spec['loser_metrics']
    task  = task_spec['task']
    exp_id = f'EXP-{rng.randint(1000,9999)}'
    result_file = f'/workspace/results_{exp_id.lower()}.json'

    trace = [
        {'role':'system',
         'content':'You are a research agent. Always cite evidence for decisions. Preserve experiment IDs and file paths.'},
        {'role':'user', 'content': task},
    ]

    trace.append({
        'role':'assistant',
        'content':f'I will search for evidence on each option: {", ".join(opts)}.',
        'tool_call':{'name':'search','args':{'query':f'{task} comparison evaluation 2024'}},
    })
    search_result = (
        f'{opts[0]}: {json.dumps(win_m)}. '
        f'{opts[1]}: {json.dumps(los_m[0])}. '
        f'{opts[2]}: {json.dumps(los_m[1])}. '
        + _make_noise(noise_level)
    )
    trace.append({'role':'tool','name':'search','content':search_result})

    trace.append({
        'role':'assistant',
        'content':f'Reading detailed report for {win}.',
        'tool_call':{'name':'read_report','args':{'target':win,'experiment':exp_id}},
    })
    metrics_str = ' '.join(f'{k}={v}' for k,v in win_m.items())
    trace.append({
        'role':'tool','name':'read_report',
        'content':(
            f'Report for {win}. Experiment: {exp_id}. '
            f'Metrics: {metrics_str}. '
            f'Output saved to {result_file}. '
            + _make_noise(noise_level)
        ),
    })

    trace.append({
        'role':'assistant',
        'content':f'Comparing all options on key metrics.',
        'tool_call':{'name':'compare','args':{'options':opts,'metric':'primary'}},
    })
    compare_content = f'Winner: {win} ({metrics_str}). '
    for i,opt in enumerate([o for o in opts if o!=win]):
        compare_content += f'{opt}: {json.dumps(los_m[min(i,len(los_m)-1)])}. '
    compare_content += _make_noise(noise_level)
    trace.append({'role':'tool','name':'compare','content':compare_content})

    metrics_cited = ', '.join(f'{k}={v}' for k,v in list(win_m.items())[:3])
    trace.append({
        'role':'assistant',
        'content':(
            f'{win} is the clear winner. '
            f'Evidence: {metrics_cited}. '
            f'Experiment: {exp_id}. Results: {result_file}. '
            f'Recommend implementing {win} first.'
        ),
    })

    trace.append({
        'role':'user',
        'content':f'Confirm the recommendation and ensure {exp_id} and {result_file} are preserved.',
    })

    trace.append({
        'role':'assistant',
        'content':(
            f'Confirmed. Recommendation: {win}. '
            f'Experiment ID {exp_id} and file {result_file} are preserved. '
            f'Key evidence: {metrics_cited}.'
        ),
    })

    return trace

print("Synthetic trace generator ready ✓")


# ================================================================
# PART 7: BASELINE COMPRESSORS
# (so "your method" has real baselines to beat)
# ================================================================

# ── CELL 12: Baseline Compressors ────────────────────────────

def compress_tail_truncation(messages, cfg=CCFG):
    """Recency-only baseline: keep system prompt + walk backwards
    from the end, keeping whole messages until budget runs out."""
    orig_toks = sum(_tok(_get_text(m)) for m in messages)
    budget = max(1, int(orig_toks * (1 - cfg.TARGET_REDUCTION)))

    sys_msgs = [(i, m) for i, m in enumerate(messages) if m.get('role') == 'system']
    rest     = [(i, m) for i, m in enumerate(messages) if m.get('role') != 'system']

    kept = {}
    used = 0
    for i, m in sys_msgs:
        mc = dict(m)
        toks = TOKENIZER.encode(m.get('content', ''))
        if len(toks) > cfg.SYSTEM_MAX_TOKENS:
            mc['content'] = TOKENIZER.decode(toks[:cfg.SYSTEM_MAX_TOKENS]) + '[…]'
        mc['_orig_idx'] = i
        kept[i] = mc
        used += _tok(mc.get('content', ''))

    for i, m in reversed(rest):
        t = _tok(_get_text(m))
        if used + t > budget and kept:
            continue
        kept[i] = dict(m, _orig_idx=i)
        used += t

    return [kept[i] for i in sorted(kept)]


def compress_random_drop(messages, cfg=CCFG, seed=0):
    """No-relevance-signal baseline: random sentence-level drop
    under the same budget."""
    rng = random.Random(seed)
    orig_toks = sum(_tok(_get_text(m)) for m in messages)
    budget = max(1, int(orig_toks * (1 - cfg.TARGET_REDUCTION)))
    n = len(messages)
    protected = {i for i, m in enumerate(messages)
                 if m.get('role') == 'system' or i >= n - cfg.KEEP_LAST_K}
    protected_toks = sum(_tok(_get_text(messages[i])) for i in protected)
    remaining = max(0, budget - protected_toks)

    pool = []
    for i, m in enumerate(messages):
        if i in protected:
            continue
        for s in _split_sents(_get_text(m), cfg.MIN_SENT_TOKENS):
            pool.append((s, i))
    rng.shuffle(pool)

    chosen, used = [], 0
    for s, i in pool:
        t = _tok(s)
        if used + t <= remaining:
            chosen.append((s, i))
            used += t

    msg_sents = {}
    for s, i in chosen:
        msg_sents.setdefault(i, []).append(s)

    out = []
    for i, m in enumerate(messages):
        if i in protected:
            mc = dict(m)
            if m.get('role') == 'system':
                toks = TOKENIZER.encode(m.get('content', ''))
                if len(toks) > cfg.SYSTEM_MAX_TOKENS:
                    mc['content'] = TOKENIZER.decode(toks[:cfg.SYSTEM_MAX_TOKENS]) + '[…]'
            mc['_orig_idx'] = i
            out.append(mc)
        elif i in msg_sents:
            mc = dict(m)
            mc['content'] = ' '.join(msg_sents[i])
            mc['_orig_idx'] = i
            out.append(mc)
    return out


def compress_plain_mmr(messages, cfg=CCFG):
    """Ablation: MMR with no artifact-budget carve-out, to isolate
    how much of your method's edge comes from relevance scoring vs.
    the explicit artifact-preservation reserve."""
    cfg2 = copy.deepcopy(cfg)
    cfg2.ARTIFACT_BUDGET_PCT = 0.0
    return compress(messages, cfg2)


def compress_mmr_unprotected(messages, cfg=CCFG):
    """Ablation: identical to Shaurya DDR EXCEPT decision-bearing
    messages are NOT structurally protected and get no evidence
    boost — i.e. this IS the original (buggy) behavior, kept as a
    baseline specifically to demonstrate what the protection fix is
    worth. Expect this to reproduce the 'search/read_report/compare
    silently vanish' failure mode."""
    cfg2 = copy.deepcopy(cfg)
    cfg2.PROTECT_TOOL_CALLS = False
    cfg2.PROTECT_JUDGMENTS  = False
    cfg2.EVIDENCE_BOOST     = 0.0
    return compress(messages, cfg2)


def compress_extractive_tfidf(messages, cfg=CCFG):
    """Classic extractive-summarization baseline: TF-IDF sentence
    centrality (no task embedding, no decision awareness, no
    artifact budget) — the kind of generic summarizer a reviewer
    would ask 'are you actually better than this?' about."""
    orig_toks = sum(_tok(_get_text(m)) for m in messages)
    budget = max(1, int(orig_toks * (1 - cfg.TARGET_REDUCTION)))
    n = len(messages)
    protected = {i for i, m in enumerate(messages)
                 if m.get('role') == 'system' or i >= n - cfg.KEEP_LAST_K}
    protected_toks = sum(_tok(_get_text(messages[i])) for i in protected)
    remaining = max(0, budget - protected_toks)

    pool = []
    for i, m in enumerate(messages):
        if i in protected:
            continue
        for s in _split_sents(_get_text(m), cfg.MIN_SENT_TOKENS):
            pool.append((s, i))

    if pool:
        sents = [s for s, _ in pool]
        terms = [re.findall(r'[a-z0-9]+', s.lower()) for s in sents]
        df = Counter()
        for t in terms:
            df.update(set(t))
        N = len(sents)
        def tfidf_vec(toks):
            tf = Counter(toks)
            return {w: tf[w] * math.log((N + 1) / (df[w] + 1)) for w in tf}
        vecs = [tfidf_vec(t) for t in terms]
        def cos_sparse(a, b):
            common = set(a) & set(b)
            num = sum(a[w] * b[w] for w in common)
            da = math.sqrt(sum(v * v for v in a.values())) or 1e-9
            db = math.sqrt(sum(v * v for v in b.values())) or 1e-9
            return num / (da * db)
        centrality = [sum(cos_sparse(vecs[i], vecs[j]) for j in range(N) if j != i) for i in range(N)]
        ranked = sorted(range(N), key=lambda i: centrality[i], reverse=True)

        chosen, used = [], 0
        for i in ranked:
            t = _tok(sents[i])
            if used + t <= remaining:
                chosen.append(i)
                used += t
        chosen = sorted(chosen)
        msg_sents = {}
        for i in chosen:
            s, mi = pool[i]
            msg_sents.setdefault(mi, []).append(s)
    else:
        msg_sents = {}

    out = []
    for i, m in enumerate(messages):
        if i in protected:
            out.append(dict(m, _orig_idx=i))
        elif i in msg_sents:
            mc = dict(m)
            mc['content'] = ' '.join(msg_sents[i])
            mc['_orig_idx'] = i
            out.append(mc)
    return out


def compress_llm_summary(messages, cfg=CCFG, model='mistral-small-latest'):
    """LLM-summarization baseline: a single Mistral call rewrites
    the non-protected middle of the trace into a free-text summary
    at the target token budget. Strong on fluency/SP, but has no
    structural guarantee that any specific decision or tool call
    survives — useful exactly because it's the strongest-sounding
    'just summarize it' alternative."""
    n = len(messages)
    protected_idx = {i for i, m in enumerate(messages)
                      if m.get('role') == 'system' or i >= n - cfg.KEEP_LAST_K}
    middle = [(i, m) for i, m in enumerate(messages) if i not in protected_idx]
    if not middle:
        return [dict(m, _orig_idx=i) for i, m in enumerate(messages)]

    orig_toks = sum(_tok(_get_text(m)) for _, m in middle)
    budget = max(1, int(orig_toks * (1 - cfg.TARGET_REDUCTION)))
    raw = '\n'.join(f'[{i}] {m.get("role","?").upper()}: {_get_text(m)}' for i, m in middle)

    try:
        resp = MISTRAL.chat.complete(
            model=model,
            messages=[
                {'role': 'system', 'content':
                    f'Summarize the following agent trace turns into roughly {budget} tokens. '
                    f'Preserve every concrete fact, number, file path, and ID verbatim. Plain text only.'},
                {'role': 'user', 'content': raw},
            ],
            temperature=0.0, max_tokens=min(1024, budget * 2 + 50),
        )
        summary = resp.choices[0].message.content.strip()
    except Exception as e:
        summary = f'[summary failed: {e}]'

    out = []
    for i, m in enumerate(messages):
        if i in protected_idx:
            out.append(dict(m, _orig_idx=i))
    out.append({'role': 'tool', 'name': 'summary', 'content': summary, '_orig_idx': -1})
    out.sort(key=lambda mc: mc['_orig_idx'] if mc['_orig_idx'] != -1 else 10**9)
    return out


BASELINES = {
    'tail_truncation':   lambda msgs, **kw: compress_tail_truncation(msgs),
    'random_drop':       lambda msgs, **kw: compress_random_drop(msgs, seed=kw.get('seed', 0)),
    'plain_mmr':         lambda msgs, **kw: compress_plain_mmr(msgs),
    'extractive_tfidf':  lambda msgs, **kw: compress_extractive_tfidf(msgs),
    'llm_summary':       lambda msgs, **kw: compress_llm_summary(msgs),
    'mmr_unprotected':   lambda msgs, **kw: compress_mmr_unprotected(msgs),  # = old buggy behavior
    'Shaurya DDR':       lambda msgs, **kw: compress(msgs),
}

print("Baseline compressors ready ✓  (tail_truncation, random_drop, plain_mmr, Shaurya DDR)")


# ================================================================
# PART 8: REASONING CHAIN INTEGRITY (RCI)
#
# DRR tests each decision in isolation by asking a fresh LLM "what
# did the agent decide at step N?" — an LLM can often guess the
# right answer from a single leaked conclusion sentence even with
# zero supporting evidence present. RCI is a structural check that
# catches this: it builds a dependency graph (decision -> evidence
# artifact -> the turn that first introduced it) from the ORIGINAL
# trace, then checks whether compression preserved those specific
# edges — not just whether the artifact string survives somewhere.
# ================================================================

# ── CELL 13: RCI ──────────────────────────────────────────────

def build_dependency_graph(messages: List[Dict], decisions: List[Dict]) -> List[Dict]:
    first_seen: Dict[str, int] = {}
    for i, m in enumerate(messages):
        arts = _artifacts(_get_text(m))
        for k in ('paths', 'ids'):
            for a in arts[k]:
                if a not in first_seen:
                    first_seen[a] = i

    edges = []
    for dec_idx, d in enumerate(decisions):
        cited = set(d['artifacts'].get('paths', []) + d['artifacts'].get('ids', []))
        for a in cited:
            origin = first_seen.get(a)
            if origin is not None and origin < d['msg_idx']:
                edges.append({
                    'to_decision_idx':  dec_idx,
                    'artifact':         a,
                    'origin_msg_idx':   origin,
                    'decision_msg_idx': d['msg_idx'],
                })
    return edges


def compute_rci(messages: List[Dict], compressed: List[Dict], decisions: List[Dict]) -> Dict:
    """RCI = fraction of original decision -> evidence dependency
    edges still satisfiable in the compressed trace. Stricter than
    artifact retention: a compressor can keep 100% of artifacts in
    aggregate while still severing the link between a *specific*
    decision and the evidence it actually cited."""
    edges = build_dependency_graph(messages, decisions)
    if not edges:
        return {'RCI': None, 'edges_total': 0, 'edges_preserved': 0, 'edges': []}

    comp_text = ' '.join(_get_text(m) for m in compressed)
    preserved = 0
    detail = []
    for e in edges:
        ok = e['artifact'] in comp_text
        preserved += int(ok)
        detail.append({**e, 'preserved': ok})

    return {
        'RCI':             round(preserved / len(edges), 4),
        'edges_total':     len(edges),
        'edges_preserved': preserved,
        'edges':           detail,
    }

print("RCI (Reasoning Chain Integrity) ready ✓")


# ── CELL 13b: Decision-to-decision dependency chains ─────────
#
# build_dependency_graph links a decision to the raw artifact it
# cited. That's enough to measure "did the evidence string survive"
# but not "did the upstream DECISION that produced that evidence
# survive" — e.g. did the 'compare' step's own tool_call message
# (not just the word "CompressMMR" somewhere in the text) make it
# through compression. attach_dependencies makes that link explicit
# (action,target,evidence,rationale,dependency) instead of the
# flatter (action,target,rationale) used for matching.

def attach_dependencies(messages: List[Dict], decisions: List[Dict]) -> List[Dict]:
    """For each decision, find which EARLIER DECISION (not just
    which raw turn) first introduced each artifact it cites. Adds
    'depends_on' (list of decision_idx) to each decision dict."""
    artifact_origin_decision: Dict[str, int] = {}
    for dec_idx, d in enumerate(decisions):
        produced = set(d['artifacts'].get('paths', []) + d['artifacts'].get('ids', []))
        for a in produced:
            if a not in artifact_origin_decision:
                artifact_origin_decision[a] = dec_idx

    for dec_idx, d in enumerate(decisions):
        cited = set(d['artifacts'].get('paths', []) + d['artifacts'].get('ids', []))
        deps = sorted({
            artifact_origin_decision[a] for a in cited
            if a in artifact_origin_decision and artifact_origin_decision[a] < dec_idx
        })
        d['depends_on'] = deps
    return decisions


def compute_chain_rci(messages: List[Dict], compressed: List[Dict], decisions: List[Dict]) -> Dict:
    """Chain-level survival: a decision chain (e.g. search -> read_report
    -> compare -> recommend) is intact only if EVERY decision in the
    chain still has its tool_call/turn present in the compressed
    trace AND every depends_on edge it relies on is still
    satisfiable. This is what 'did the reasoning trajectory survive,
    not just the outcome' actually means, operationally.
    """
    decisions = attach_dependencies(messages, decisions)
    comp_idx_present = {cm['_orig_idx'] for cm in compressed if '_orig_idx' in cm}

    chains_total = chains_intact = 0
    detail = []
    for dec_idx, d in enumerate(decisions):
        if not d['depends_on']:
            continue
        chains_total += 1
        own_present = d['msg_idx'] in comp_idx_present
        deps_present = all(decisions[dep]['msg_idx'] in comp_idx_present for dep in d['depends_on'])
        intact = own_present and deps_present
        chains_intact += int(intact)
        detail.append({
            'decision_idx': dec_idx, 'depends_on': d['depends_on'],
            'own_present': own_present, 'deps_present': deps_present, 'intact': intact,
        })

    if chains_total == 0:
        return {'chain_RCI': None, 'chains_total': 0, 'chains_intact': 0, 'detail': detail}
    return {
        'chain_RCI':    round(chains_intact / chains_total, 4),
        'chains_total': chains_total,
        'chains_intact': chains_intact,
        'detail':       detail,
    }

print("Decision dependency chains + chain-level RCI ready ✓")


# ================================================================
# PART 9: CONCLUSION-LEAKAGE CONTROL
#
# Diagnostic, not a real compressor: keep ONLY the final
# judgment/confirmation sentences and strip all supporting
# evidence. If DRR stays high here while RCI collapses, your
# reproducer was pattern-matching a leaked conclusion rather than
# reasoning from preserved evidence — report DRR alongside RCI from
# then on, not alone.
# ================================================================

# ── CELL 14: Leakage Control ──────────────────────────────────

def compress_conclusion_only(messages: List[Dict], decisions: List[Dict]) -> List[Dict]:
    keep_idx = {d['msg_idx'] for d in decisions if d['type'] in ('judgment', 'confirmation')}
    out = []
    for i, m in enumerate(messages):
        if m.get('role') in ('system', 'user'):
            out.append(dict(m, _orig_idx=i))
        elif i in keep_idx:
            mc = dict(m)
            mc.pop('tool_call', None)
            mc['_orig_idx'] = i
            out.append(mc)
    return out


def run_leakage_control(messages: List[Dict], verbose: bool = True) -> Dict:
    decisions = extract_decisions(messages)
    if not decisions:
        return {}
    leaked = compress_conclusion_only(messages, decisions)

    results = []
    for d in decisions:
        repro = reproduce_decision(leaked, d)
        match = match_decision(d, repro)
        results.append({'original': d, 'reproduced': repro, 'match': match})

    drr_soft = float(np.mean([r['match']['decision_score'] for r in results]))
    rci = compute_rci(messages, leaked, decisions)

    out = {
        'DRR_soft_on_leaked_trace': round(drr_soft, 4),
        'RCI_on_leaked_trace':      rci['RCI'],
        'gap':                      None if rci['RCI'] is None else round(drr_soft - rci['RCI'], 4),
    }

    if verbose:
        print(f"  DRR on conclusion-only trace : {out['DRR_soft_on_leaked_trace']:.4f}")
        print(f"  RCI on conclusion-only trace : {out['RCI_on_leaked_trace']}")
        if out['gap'] is not None and out['gap'] > 0.4:
            print(f"  ⚠ GAP = {out['gap']:.3f} — DRR is largely guess-driven on this trace.")
        elif out['gap'] is not None:
            print(f"  GAP = {out['gap']:.3f} — DRR and RCI move together; less guess risk.")

    return out

print("Conclusion-leakage control ready ✓")


# ================================================================
# PART 10: METHOD COMPARISON BENCHMARK
# ================================================================

# ── CELL 15: Method Comparison ─────────────────────────────────

def _recommendation_match(results: List[Dict]) -> float:
    rel = [r for r in results if r['original']['type'] in ('judgment', 'confirmation')]
    if not rel:
        return float('nan')
    return float(np.mean([
        0.5 * r['match']['action_score'] + 0.5 * (r['match']['target_score'] or 0.0) for r in rel
    ]))

def _evidence_match(results: List[Dict]) -> float:
    return float(np.mean([r['match']['rationale_score'] for r in results])) if results else float('nan')


def run_method_comparison(
    n_traces_per_task: int = 2,
    noise_levels: List[int] = [3],
    methods: Dict[str, Any] = None,
    verbose: bool = True,
) -> Dict[str, Dict]:
    """Method | Compression% | Recommendation Match | Evidence Match
    | Artifact Retention | CRR | RCI — same traces, same budget.

    Cost note: this calls reproduce_decision() once per decision per
    method per trace. Defaults: 5 tasks x 2 seeds x ~3 decisions x
    4 methods ≈ 120 Mistral calls. Tune args to control cost.
    """
    methods = methods or BASELINES
    agg = {name: {'compression': [], 'rec_match': [], 'evid_match': [],
                   'art_ret': [], 'CRR': [], 'RCI': []} for name in methods}

    for task_idx, task_spec in enumerate(TASKS):
        for noise in noise_levels:
            for seed in range(n_traces_per_task):
                trace = generate_trace(task_spec, noise_level=noise, rng_seed=seed * 100 + task_idx)
                decisions = extract_decisions(trace)
                if not decisions:
                    continue
                orig_toks = sum(_tok(_get_text(m)) for m in trace)

                for name, fn in methods.items():
                    compressed = fn(trace, seed=seed)
                    comp_toks = sum(_tok(_get_text(m)) for m in compressed)
                    reduction = 100 * (1 - comp_toks / max(1, orig_toks))

                    results = []
                    for d in decisions:
                        repro = reproduce_decision(compressed, d)
                        results.append({'original': d, 'reproduced': repro,
                                         'match': match_decision(d, repro)})

                    drr_bin = float(np.mean([r['match']['reproduced'] for r in results]))
                    legacy = _legacy_metrics(trace, compressed)
                    rci = compute_rci(trace, compressed, decisions)

                    agg[name]['compression'].append(reduction)
                    agg[name]['rec_match'].append(_recommendation_match(results))
                    agg[name]['evid_match'].append(_evidence_match(results))
                    agg[name]['art_ret'].append(legacy['art_ret'])
                    agg[name]['CRR'].append(drr_bin)
                    if rci['RCI'] is not None:
                        agg[name]['RCI'].append(rci['RCI'])

    summary = {}
    for name, vals in agg.items():
        summary[name] = {k: (float(np.nanmean(v)) if v else float('nan')) for k, v in vals.items()}

    if verbose:
        print('=' * 78)
        print('METHOD COMPARISON  (same compression budget, same traces)')
        print('=' * 78)
        print(f'{"Method":>16}  {"Compress%":>9}  {"RecMatch":>9}  {"EvidMatch":>9}  '
              f'{"ArtRet":>7}  {"CRR":>6}  {"RCI":>6}')
        print('-' * 78)
        for name, row in summary.items():
            print(f'{name:>16}  {row["compression"]:>8.1f}%  {row["rec_match"]:>8.2%}  '
                  f'{row["evid_match"]:>8.2%}  {row["art_ret"]:>6.2%}  '
                  f'{row["CRR"]:>5.2%}  {row["RCI"]:>5.2%}')
        print('=' * 78)
        if 'Shaurya DDR' in summary and 'tail_truncation' in summary:
            d_crr = summary['Shaurya DDR']['CRR'] - summary['tail_truncation']['CRR']
            d_rci = summary['Shaurya DDR']['RCI'] - summary['tail_truncation']['RCI']
            print(f'\nYour Method vs tail_truncation:  ΔCRR = {d_crr:+.2%}   ΔRCI = {d_rci:+.2%}')

    return summary

print("Method comparison benchmark ready ✓")


# ================================================================
# PART 11: FULL BENCHMARK OVER N TRACES (SP vs DRR correlation)
# ================================================================

# ── CELL 16: Benchmark Runner ────────────────────────────────

def run_benchmark(
    n_traces_per_task: int = 3,
    noise_levels:      List[int] = [1, 3, 5],
    compress_cfg:      CompressorCfg = CCFG,
    verbose:           bool = False,
) -> List[Dict]:
    """DRR benchmark across all task specs x noise levels.
    Default: 5 x 3 x 3 = 45 traces."""
    all_results = []
    total = len(TASKS) * n_traces_per_task * len(noise_levels)
    done  = 0

    print('='*65)
    print(f'DRR BENCHMARK  —  {total} traces')
    print('='*65)
    print(f'{"Trace":>5}  {"Task":>5}  {"Noise":>5}  {"DRR_soft":>9}  '
          f'{"DRR_bin":>8}  {"SP":>7}  {"Red%":>6}  {"ArtRet":>8}')
    print('-'*65)

    for task_idx, task_spec in enumerate(TASKS):
        for noise in noise_levels:
            for seed in range(n_traces_per_task):
                trace = generate_trace(task_spec, noise_level=noise, rng_seed=seed*100+task_idx)
                result = compute_drr(trace, compress, compress_cfg, verbose=verbose)
                result['task_idx']   = task_idx
                result['noise']      = noise
                result['seed']       = seed
                result['task_label'] = task_spec['task'][:35]
                all_results.append(result)
                done += 1

                drr_s = result['DRR_soft']   if result['DRR_soft']   is not None else float('nan')
                drr_b = result['DRR_binary'] if result['DRR_binary'] is not None else float('nan')
                print(f'{done:>5}  {task_idx:>5}  {noise:>5}  '
                      f'{drr_s:>9.4f}  {drr_b:>7.2%}  '
                      f'{result.get("SP",0):>7.4f}  '
                      f'{result.get("reduction",0):>5.1f}%  '
                      f'{result.get("art_ret",0):>7.2%}')

    print()
    _print_benchmark_summary(all_results)
    return all_results

def _print_benchmark_summary(results: List[Dict]) -> None:
    valid = [r for r in results if r['DRR_soft'] is not None]
    if not valid:
        print('No valid results.')
        return

    drr_s  = [r['DRR_soft']   for r in valid]
    drr_b  = [r['DRR_binary'] for r in valid]
    sps    = [r.get('SP',0)   for r in valid]
    reds   = [r.get('reduction',0) for r in valid]
    arts   = [r.get('art_ret',0)   for r in valid]

    sp_arr  = np.array(sps)
    drr_arr = np.array(drr_s)
    if len(sp_arr) > 1 and sp_arr.std() > 0 and drr_arr.std() > 0:
        corr = float(np.corrcoef(sp_arr, drr_arr)[0,1])
    else:
        corr = float('nan')

    deceptive = [r for r in valid if r.get('SP',0) > 0.85 and r['DRR_soft'] < 0.50]

    print('='*65)
    print('BENCHMARK SUMMARY')
    print('='*65)
    print(f'  Traces evaluated          : {len(valid)}')
    print(f'  Mean DRR  (soft)          : {np.mean(drr_s):.4f}')
    print(f'  Mean DRR  (binary)        : {np.mean(drr_b):.2%}')
    print(f'  Mean SP                   : {np.mean(sps):.4f}')
    print(f'  Mean reduction            : {np.mean(reds):.1f}%')
    print(f'  Mean artifact retention   : {np.mean(arts):.2%}')
    print(f'  Pearson(SP, DRR_soft)     : {corr:+.4f}')
    if not math.isnan(corr):
        if abs(corr) < 0.3:
            print(f'  → SP and DRR are largely UNCORRELATED.')
        elif corr < 0:
            print(f'  → SP and DRR are NEGATIVELY correlated.')
        else:
            print(f'  → SP and DRR show some correlation ({corr:.2f}).')
    print(f'  "Deceptive" cases (SP>0.85 but DRR<0.50): {len(deceptive)}')
    for r in deceptive[:3]:
        print(f'    task={r["task_label"][:30]}  noise={r["noise"]}  SP={r.get("SP",0):.3f}  DRR={r["DRR_soft"]:.3f}')

    noise_levels = sorted(set(r['noise'] for r in valid))
    if len(noise_levels) > 1:
        print(f'\n  DRR by noise level:')
        for nl in noise_levels:
            subset = [r for r in valid if r['noise']==nl]
            m_drr  = np.mean([r['DRR_soft'] for r in subset])
            m_sp   = np.mean([r.get('SP',0) for r in subset])
            print(f'    noise={nl}:  DRR={m_drr:.4f}  SP={m_sp:.4f}  n={len(subset)}')
    print('='*65)


# ================================================================
# PART 12: DEMO
# ================================================================

# ── CELL 17: Demo on one trace, ceiling vs real compression ───

print('\nGenerating demo trace...')
DEMO_TRACE = generate_trace(TASKS[0], noise_level=3, rng_seed=42)

print(f'Trace has {len(DEMO_TRACE)} messages:')
for i, m in enumerate(DEMO_TRACE):
    tc = f'  TOOL:{m["tool_call"]["name"]}' if m.get("tool_call") else ''
    print(f'  [{i}] {m["role"]:10s}: {_get_text(m)[:80]}{tc}')

print('\n=== CEILING (zero compression) — best any compressor could score ===')
ceiling = compute_drr(DEMO_TRACE, compress_identity, verbose=True)

print(f'\n=== YOUR METHOD (target {CCFG.TARGET_REDUCTION:.0%} reduction — see actual achieved below) ===')
real = compute_drr(DEMO_TRACE, compress, verbose=True)

print(f"\nCeiling DRR_soft : {ceiling['DRR_soft']:.4f}")
print(f"Real DRR_soft    : {real['DRR_soft']:.4f}")
print(f"Gap attributable to compression (not extraction bugs): {ceiling['DRR_soft'] - real['DRR_soft']:+.4f}")


# ── CELL 18: Robustness suite — multiple traces, real baselines ─
#
# CELL 17 above is a single illustrative trace, kept for step-by-step
# debugging. It is NOT evidence of anything by itself — one trace
# proves nothing about whether the compressor generalises, and it
# has no competing method to beat. This cell is the actual evidence:
# every TASK spec x several noise levels x several seeds, scored
# against six baselines under the identical token budget.
#
# Cost note: each reproduce_decision() call is one Mistral request.
#   run_benchmark(n_traces_per_task=2, noise_levels=[1,3,5])
#     = 5 tasks x 2 seeds x 3 noise x ~5 decisions  ≈ 150 calls
#   run_method_comparison(n_traces_per_task=2, noise_levels=[3])
#     = 5 tasks x 2 seeds x ~5 decisions x 7 methods ≈ 350 calls
# Lower n_traces_per_task / noise_levels / BASELINES below to cut
# cost; raise them for a more statistically solid result.

print("\n" + "="*65)
print("ROBUSTNESS SUITE — multiple traces, multiple noise levels")
print("="*65)
benchmark_results = run_benchmark(
    n_traces_per_task=2,
    noise_levels=[1, 3, 5],
    verbose=False,
)

print("\n" + "="*78)
print("BASELINE COMPARISON — Shaurya DDR vs. 6 alternative compressors")
print("="*78)
comparison = run_method_comparison(
    n_traces_per_task=2,
    noise_levels=[3],
    methods=BASELINES,   # tail_truncation, random_drop, plain_mmr,
                          # extractive_tfidf, llm_summary, mmr_unprotected, Shaurya DDR
)

# Make the "is this actually better, not just smaller" case explicit:
# same compression budget, compare CRR (decision reproducibility,
# binary) and RCI (evidence-chain integrity) head to head.
print("\n" + "="*65)
print("YOUR METHOD vs EACH BASELINE  (same token budget)")
print("="*65)
ym = comparison.get('Shaurya DDR', {})
for name, row in comparison.items():
    if name == 'Shaurya DDR':
        continue
    d_crr = ym.get('CRR', float('nan')) - row.get('CRR', float('nan'))
    d_rci = ym.get('RCI', float('nan')) - row.get('RCI', float('nan'))
    print(f'  vs {name:>16}:  ΔCRR = {d_crr:+.2%}   ΔRCI = {d_rci:+.2%}   '
          f'(baseline compression={row.get("compression", float("nan")):.1f}%)')

# Optional — uncomment for the conclusion-leakage diagnostic across
# all 5 task families instead of just one trace:
print("\n--- Conclusion-leakage control, one trace per task ---")
for task_spec in TASKS:
     trace = generate_trace(task_spec, noise_level=3, rng_seed=7)
     print(f"\nTask: {task_spec['task']}")
     run_leakage_control(trace)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF authenticated ✓


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Models + Mistral client loaded ✓
Compressor ready ✓
Decision extractor ready ✓
Decision reproducer ready ✓
Decision matcher ready ✓
DRR pipeline ready ✓
Synthetic trace generator ready ✓
Baseline compressors ready ✓  (tail_truncation, random_drop, plain_mmr, Shaurya DDR)
RCI (Reasoning Chain Integrity) ready ✓
Decision dependency chains + chain-level RCI ready ✓
Conclusion-leakage control ready ✓
Method comparison benchmark ready ✓

Generating demo trace...
Trace has 11 messages:
  [0] system    : You are a research agent. Always cite evidence for decisions. Preserve experimen
  [1] user      : Choose the best compression algorithm for agent traces.
  [2] assistant : I will search for evidence on each option: CompressMMR, BudgetComp, SelectiveCtx  TOOL:search
  [3] tool      : CompressMMR: {"SP": 0.91, "task_success": 0.82, "compression": 0.73}. BudgetComp
  [4] assistant : Reading detailed report for CompressMMR. {"args": {"experiment": "EXP-2824", "ta  TOOL:read_report
  [5] tool    